Import modules

In [ ]:
import os, multiprocessing as mp
cores = mp.cpu_count()
os.environ["TF_NUM_INTRAOP_THREADS"] = str(cores)  # parallel within ops
os.environ["TF_NUM_INTEROP_THREADS"] = "2"         # parallel across ops
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "1"          # Intel-optimized kernels

import tensorflow as tf
tf.config.optimizer.set_jit(True)  # try XLA on CPU
gpus = tf.config.experimental.list_physical_devices("GPU")
print("TF:", tf.__version__)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs detected: {len(gpus)}")
    except RuntimeError as e:
        print("Error setting memory growth:", e)
else:
    print("No GPU detected")

import cv2
from tensorflow.keras.utils import normalize
from skimage.morphology import remove_small_holes, remove_small_objects, binary_closing, disk, binary_dilation
from skimage.measure import label, regionprops

import math
import matplotlib.pyplot as plt

from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall

In [ ]:
# ==============================
# Post-processing (optional)
# ==============================
def clean_mask_with_contrast(P, img, *,
                             thr=0.5,
                             hole_area=600,
                             min_blob=400, max_blob=6000,
                             prob_bg=0.30,
                             contrast_thr=0.10,
                             ring_r=3, close_r=3, min_fg=150):
    m0 = P > thr
    m1 = binary_closing(m0, disk(close_r)) if close_r else m0
    m_filled = remove_small_holes(m1, area_threshold=hole_area)

    added = m_filled & (~m1)
    lab = label(added, connectivity=1)
    restore = np.zeros_like(m_filled, dtype=bool)

    rng = float(img.max() - img.min() + 1e-8)
    for reg in regionprops(lab):
        a = reg.area
        if not (min_blob <= a <= max_blob):  # size gate in pixels
            continue
        rr, cc = zip(*reg.coords)
        mean_p = float(np.mean(P[rr, cc]))
        if mean_p > prob_bg:  # looks like foreground grid → don't restore
            continue
        ring = binary_dilation(np.array(reg.image, dtype=bool), disk(ring_r))
        # paste ring in full coords
        hole_mask = np.zeros_like(m_filled, bool); hole_mask[rr, cc] = True
        ring_full = binary_dilation(hole_mask, disk(ring_r)) & ~hole_mask
        if ring_full.sum() == 0: continue
        I_in, I_out = img[hole_mask].mean(), img[ring_full].mean()
        contrast = abs(I_in - I_out) / rng
        if contrast >= contrast_thr:
            restore[rr, cc] = True

    m_final = m_filled & (~restore)
    m_final = remove_small_objects(m_final, min_size=min_fg)
    return m_final

# ==============================
# Patch-based prediction
# ==============================
def predict_full_image_prob(model, image, patch_size=256):
    # If you prefer to infer from the model instead of hardcoding:
    # h_in, w_in = model.input_shape[1:3]; patch_size = h_in or patch_size

    H, W = image.shape[:2]
    prob_map  = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            # take whatever fits at the border
            patch = image[i:i+patch_size, j:j+patch_size]
            ph, pw = patch.shape[:2]

            # normalize to float32 [0,1], add channel dim
            pnorm = normalize(np.asarray(patch, dtype=np.float32), axis=1)[..., np.newaxis]

            # pad to (patch_size, patch_size, 1) for the model
            if ph < patch_size or pw < patch_size:
                canvas = np.zeros((patch_size, patch_size, 1), dtype=np.float32)
                canvas[:ph, :pw, 0] = pnorm[..., 0]
            else:
                canvas = pnorm

            pin = canvas[np.newaxis, ...]  # (1, H, W, 1)

            # predict -> (1, patch_size, patch_size, 1)
            out = model.predict(pin, verbose=0)
            if isinstance(out, (list, tuple)): out = out[0]
            pred_full = np.asarray(out, dtype=np.float32).squeeze()
            if pred_full.ndim == 3: pred_full = pred_full[..., 0]  # (patch_size, patch_size)

            # crop back to the original patch region (ph, pw)
            pred = pred_full[:ph, :pw]

            # write into full canvas
            prob_map[i:i+ph, j:j+pw]  += pred
            count_map[i:i+ph, j:j+pw] += 1.0

    count_map[count_map == 0] = 1.0
    prob_map /= count_map
    return np.clip(prob_map, 0, 1).astype(np.float32)



# ==============================
# Per-image, multi-model panel
# ==============================
def visualize_models_on_image(models, img, thr=0.6, use_post=True, pp_kwargs=None,
                              show_probs=False, gt=None, savepath=None, title=""):
    """
    models: dict[name->keras.Model]
    img   : (H,W) grayscale
    gt    : optional ground-truth mask (H,W) in {0,1}
    """
    pp_kwargs = pp_kwargs or {}
    preds = []

    # get display base
    img_disp = np.asarray(img, dtype=np.float32)

    # compute per-model prob & mask
    for name, m in models.items():
        P = predict_full_image_prob(m, img_disp, patch_size=256)
        if use_post:
            mask = clean_mask_with_contrast(P, img_disp, thr=thr, **pp_kwargs)
        else:
            mask = P >= thr

        # metrics if GT is provided
        iou = dice = None
        if gt is not None:
            gt_b = gt > 0.5
            inter = int(np.logical_and(gt_b, mask).sum())
            union = int(np.logical_or(gt_b, mask).sum())
            iou = (inter / union) if union > 0 else 1.0
            denom = int(gt_b.sum()) + int(mask.sum())
            dice = (2.0 * inter / denom) if denom > 0 else 1.0

        preds.append((name, P, mask, iou, dice))

    # layout: image, (optional GT), then each model
    extra = 1 + (1 if gt is not None else 0)
    panels = extra + len(preds)
    cols = min(panels, 6)
    rows = math.ceil(panels / cols)
    fig = plt.figure(figsize=(4 * cols, 4 * rows))

    # 1) images image
    ax = plt.subplot(rows, cols, 1)
    ax.set_title(f"{title} raw")
    ax.imshow(img_disp, cmap="gray"); ax.axis("off")

    # 2) GT (optional)
    k = 2
    if gt is not None:
        ax = plt.subplot(rows, cols, k); k += 1
        ax.set_title("GT")
        ax.imshow(img_disp, cmap="gray")
        ax.imshow(gt > 0.5, alpha=0.4); ax.axis("off")

    # 3..) each model
    for name, P, mask, iou, dice in preds:
        ax = plt.subplot(rows, cols, k); k += 1
        tag = " +pp" if use_post else ""
        met = ""
        if iou is not None:
            met = f"\nIoU={iou:.3f} Dice={dice:.3f}"
        ax.set_title(f"{name}{tag}{met}")
        ax.imshow(img_disp, cmap="gray")
        overlay = P if show_probs else mask
        im = ax.imshow(overlay, alpha=0.4)
        ax.axis("off")
        if show_probs:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    else:
        plt.show()

# ==============================
# Directory runner
# ==============================
def run_dir_multimodel(models, img_dir, out_dir,
                       thr=0.6, use_post=True, pp_kwargs=None,
                       patterns=("*.png","*.jpg","*.jpeg","*.tif","*.tiff"),
                       show_probs=False, gt_suffix=None):
    """
    gt_suffix: if you have GT masks, set e.g. '_mask.png' to load file.stem+gt_suffix next to the image
    """
    img_dir = Path(img_dir); out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for pat in patterns:
        files.extend(sorted(img_dir.glob(pat)))

    for f in files:
        print(f"\n=== {f.name} ===")
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("  (skipped: unreadable)"); continue

        gt = None
        if gt_suffix:
            gt_path = f.with_name(f.stem + gt_suffix)
            if gt_path.exists():
                gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
                if gt is not None: gt = (gt > 127).astype(np.uint8)

        savepath = out_dir / f"{f.stem}_panel.png"
        visualize_models_on_image(
            models, img, thr=thr, use_post=use_post, pp_kwargs=pp_kwargs or {},
            show_probs=show_probs, gt=gt, savepath=str(savepath), title=f.stem
        )
        print(f"Saved {savepath}")


Load the models

In [ ]:
from pathlib import Path
from tensorflow.keras.models import load_model

# point to your checkpoint root
ckptdir = Path("outputs/checkpoints")

# (optional) if you only need predict(), skip compiling & custom_objects
def load_all_unets(ckptdir):
    models = {}
    for d in sorted(ckptdir.glob("linknet_*")):
        if d.is_dir():
            name = d.name.replace("linknet_", "")
            print(f"Loading {name} from {d}")
            models[name] = load_model(d, compile=False)  # SavedModel folder
    return models

models = load_all_unets(ckptdir)
print("Loaded:", list(models.keys()))

In [ ]:
import os, glob, numpy as np, tensorflow as tf
from types import SimpleNamespace

def _make_tflite_predictor(path, normalize=True):
    """Return object with .predict(x, verbose=0) that mimics Keras."""
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    in_det  = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    _, H, W, C = tuple(in_det['shape'])

    # quantization params (if any)
    in_scale, in_zero  = (in_det.get('quantization', (0.0, 0)) or (0.0, 0))
    out_scale, out_zero = (out_det.get('quantization', (0.0, 0)) or (0.0, 0))
    in_dtype  = in_det['dtype']
    out_dtype = out_det['dtype']

    def predict(x, verbose=0):
        x = np.asarray(x)

        # accept (H,W), (H,W,1|C), or (N,H,W,C)
        if x.ndim == 2:          # (H,W) -> (1,H,W,1)
            x = x[None, ..., None]
        elif x.ndim == 3:        # (H,W,C) -> (1,H,W,C)
            x = x[None, ...]
        elif x.ndim != 4:
            raise ValueError(f"Unsupported input shape {x.shape}")

        # force channel count to C
        if x.shape[-1] != C:
            if x.shape[-1] == 1 and C == 3:
                x = np.repeat(x, 3, axis=-1)
            else:
                x = x[..., :C]

        # resize batch to (H,W) using TF (robust for single channel)
        x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
        x_tf = tf.image.resize(x_tf, (H, W), method='bilinear')
        xx = x_tf.numpy()  # (N,H,W,C) float32

        # normalize to [0,1] if values look like 0..255 and model expects float
        if normalize and in_dtype != np.uint8 and xx.max() > 1.5:
            xx = xx / 255.0

        # If the TFLite model is quantized (uint8), quantize input
        if in_dtype == np.uint8 and in_scale and in_scale > 0:
            xx = np.clip(np.round(xx / in_scale + in_zero), 0, 255).astype(np.uint8)
        else:
            xx = xx.astype(in_dtype)

        # run
        interp.set_tensor(in_det['index'], xx)
        interp.invoke()
        pred = interp.get_tensor(out_det['index'])

        # dequantize output if needed
        if out_dtype == np.uint8 and out_scale and out_scale > 0:
            pred = (pred.astype(np.float32) - out_zero) * out_scale
        else:
            pred = pred.astype(np.float32)

        # squeeze channel if it's 1
        if pred.ndim == 4 and pred.shape[-1] == 1:
            pred = pred[..., 0]
        return pred

    return SimpleNamespace(predict=predict, name=os.path.basename(path))

def load_tflite_dir_as_models(dir_path, pattern="*.tflite", normalize=True):
    models = {}
    for p in sorted(glob.glob(os.path.join(dir_path, pattern))):
        name = os.path.splitext(os.path.basename(p))[0]
        models[name] = _make_tflite_predictor(p, normalize=normalize)
        print(f"Loaded TFLite: {name}")
    return models


In [ ]:

def parse_tflite_filename(path):
    """
    '{arch}-{backbone}_{loss}.tflite' -> (arch, backbone, loss, stem)
    Keeps full multi-part loss (e.g. 'bce_dice_focal').
    """
    stem = os.path.splitext(os.path.basename(path))[0]
    if '_' in stem:
        left, loss = stem.split('_', 1)   # keep full loss part
    else:
        left, loss = stem, ''
    if '-' in left:
        arch, backbone = left.split('-', 1)
    else:
        arch, backbone = left, ''
    return arch, backbone, loss, stem

def load_tflite_by_arch(
    dir_path,
    arch=None,
    backbones=None,
    losses=None,               # e.g. "bce"
    normalize=True,
    key_mode="auto",           # adds 'arch' mode
    one_per_arch=False,        # pick one model per architecture
    backbone_preference=None   # e.g. ["efficientnetb0","resnet34","resnet18"]
):
    if isinstance(backbones, str): backbones = {backbones}
    if isinstance(losses, str):    losses    = {losses}

    paths = sorted(glob.glob(os.path.join(dir_path, "*.tflite")))
    items = []
    for p in paths:
        a, b, l, stem = parse_tflite_filename(p)
        if arch      and a != arch:           continue
        if backbones and b not in backbones:  continue
        if losses    and l not in losses:     continue   # exact match, e.g. only 'bce'
        items.append((p, a, b, l, stem))

    # If requested, keep only one model per architecture (choose backbone by preference)
    if one_per_arch and items:
        pref = backbone_preference or []
        def rank(bb):
            return pref.index(bb) if bb in pref else len(pref)
        chosen = {}
        for p, a, b, l, s in items:
            pick = chosen.get(a)
            if pick is None or rank(b) < rank(pick[2]):
                chosen[a] = (p, a, b, l, s)
        items = list(chosen.values())

    # Decide keys shown in the figure
    if key_mode == "auto":
        key_mode = "arch" if one_per_arch or (losses and not backbones) else (
            "loss" if len({b for _,_,b,_,_ in items}) <= 1 else "backbone_loss"
        )

    models = {}
    used = set()
    def _uniq(k):
        if k not in used:
            used.add(k); return k
        i = 2
        while f"{k} ({i})" in used:
            i += 1
        k2 = f"{k} ({i})"; used.add(k2); return k2

    for p, a, b, l, stem in items:
        if   key_mode == "arch":          key = a or stem
        elif key_mode == "stem":          key = stem
        elif key_mode == "loss":          key = l or stem
        elif key_mode == "backbone_loss": key = f"{b}_{l}" if l else b
        else:                             key = stem
        key = _uniq(key)
        models[key] = _make_tflite_predictor(p, normalize=normalize)
        print(f"Loaded TFLite: {stem} -> key='{key}'")
    return models


def group_models_by_arch(dir_path, normalize=True):
    """
    Convenience: returns {arch: {stem: predictor}} for *all* .tflite files.
    """
    grouped = {}
    for p in sorted(glob.glob(os.path.join(dir_path, "*.tflite"))):
        a, b, l, stem = parse_tflite_filename(p)
        grouped.setdefault(a, {})
        grouped[a][stem] = _make_tflite_predictor(p, normalize=normalize)
        print(f"Loaded TFLite: {stem} (arch={a})")
    return grouped


In [ ]:
pp = dict(hole_area=600, min_blob=400, max_blob=6000,
          prob_bg=0.3, contrast_thr=0.10, ring_r=3, close_r=3, min_fg=150)

In [ ]:
pp

In [ ]:
tflite_dir = "/home/anvy4548/projects/crystal-recognition/training_result_tflite/"
models = load_tflite_by_arch(tflite_dir, losses='bce_dice')  # key_mode auto→"loss"
# models keys like: "bce", "bce_dice", "focal", ...
run_dir_multimodel(
    models,
    img_dir=Path("/home/anvy4548/projects/crystal-recognition/test_images"),
    out_dir=Path("outputs/panels/diffmodels/bce_dice"),
    thr=0.6,
    use_post=False,
    show_probs=False,
    gt_suffix=None,
)

In [ ]:
# Folder with your files like "unet-efficientnetb0_bce_dice.tflite", etc.
tflite_dir = "/home/anvy4548/projects/crystal-recognition/training_result_tflite/"

models = load_tflite_dir_as_models(tflite_dir)
#models = {name: type("TFLiteModel", (), {"predict": fn})() for name, fn in models.items()}

# Now plug straight into your existing pipeline:
run_dir_multimodel(
    models,
    img_dir=Path("/home/anvy4548/projects/crystal-recognition/test_images"),
    out_dir=Path("outputs/panels/v1/unet"),
    thr=0.6,
    use_post=False,
    show_probs=False,
    gt_suffix=None
)


Load the images

In [ ]:
img_root = Path("/home/anvy4548/projects/crystal-recognition/test_images")
out_root = Path("outputs/panels/linknet"); out_root.mkdir(parents=True, exist_ok=True)

pp = dict(hole_area=600, min_blob=400, max_blob=6000,
          prob_bg=0.3, contrast_thr=0.10, ring_r=3, close_r=3, min_fg=150)

run_dir_multimodel(
    models,
    img_dir=img_root,
    out_dir=out_root,
    thr=0.6,
    use_post=False,
    show_probs=False,
    gt_suffix=None
)


In [ ]:
def _get(row, *names, default=float("nan")):
    for n in names:
        if n in row and row[n] not in (None, ""):
            try:
                return float(row[n])
            except (TypeError, ValueError):
                return default
    return default

def _all_nan(xs):
    return all(isinstance(x, float) and math.isnan(x) for x in xs)

for s in summaries:
    hist = read_history_plain(Path(s["csv"]))
    if not hist:
        continue

    # epochs (fall back to 1..N if missing)
    epochs = []
    for i, r in enumerate(hist, start=1):
        try:
            e = int(r.get("epoch"))
            epochs.append(e + 1)  # CSVLogger epoch is 0-based
        except (TypeError, ValueError):
            epochs.append(i)

    # ----- Loss (always try to plot) -----
    tr_loss = [_get(r, "loss", "train_loss") for r in hist]
    va_loss = [_get(r, "val_loss") for r in hist]

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, tr_loss, label="train_loss")
    plt.plot(epochs, va_loss, label="val_loss")
    plt.title(f"{s.get('loss', 'model')} – Loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
    plt.tight_layout()
    plt.show()

    # ----- Other metrics (auto-detect: acc/prec/rec/etc.) -----
    sample_keys = set(hist[0].keys())
    # exclude non-metrics
    exclude = {"epoch", "loss", "lr"}
    base_metrics = sorted(k for k in sample_keys
                          if not k.startswith("val_") and k not in exclude)

    # Always try common accuracy aliases if not present explicitly
    if not any(k in base_metrics for k in ("acc", "accuracy")):
        base_metrics += ["acc", "accuracy"]  # harmless if missing

    seen = set()
    for m in base_metrics:
        if m in exclude or m.startswith("val_") or m in seen:
            continue
        seen.add(m)

        tr_vals = [_get(r, m) for r in hist]
        va_vals = [_get(r, f"val_{m}") for r in hist]

        # skip if both series are NaN (metric not logged)
        if _all_nan(tr_vals) and _all_nan(va_vals):
            continue

        plt.figure(figsize=(8, 4))
        plt.plot(epochs, tr_vals, label=f"train_{m}")
        plt.plot(epochs, va_vals, label=f"val_{m}")
        pretty_name = {"acc": "Accuracy", "accuracy": "Accuracy",
                       "prec": "Precision", "rec": "Recall"}.get(m, m)
        plt.title(f"{s.get('loss', 'model')} – {pretty_name}")
        plt.xlabel("epoch"); plt.ylabel(pretty_name.lower()); plt.legend()
        plt.tight_layout()
        plt.show()
